# Notebook 03 — Exploratory Data Analysis

## Objectives
This notebook answers **Business Requirement 1** — understanding which customer attributes are most associated with churn. I use correlation analysis and visualisations to explore the relationships between features and the Churn target.

* Examine the distribution of the Churn target variable
* Identify numeric features most correlated with Churn
* Visualise top features against Churn for both numeric and categorical columns
* Save all plots to `outputs/eda/` for display in the Streamlit dashboard

## Inputs
* `outputs/datasets/collection/telco_churn_cleaned.csv`

## Outputs
* `outputs/eda/churn_distribution.png`
* `outputs/eda/correlation_heatmap.png`
* `outputs/eda/tenure_vs_churn.png`
* `outputs/eda/monthlycharges_vs_churn.png`
* `outputs/eda/contract_vs_churn.png`
* `outputs/eda/techsupport_vs_churn.png`
* `outputs/eda/internetservice_vs_churn.png`

---

## 1 — Set Up the Working Directory

In [ ]:
import os

os.chdir('..') if os.path.basename(os.getcwd()) == 'jupyter_notebooks' else None
os.makedirs('outputs/eda', exist_ok=True)
print('Working directory:', os.getcwd())

## 2 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', palette='muted')

## 3 — Load Cleaned Dataset

In [ ]:
df = pd.read_csv('outputs/datasets/collection/telco_churn_cleaned.csv')
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head(3)

## 4 — Churn Distribution

Before diving into correlations I want to visualise how many customers actually churned versus stayed. This gives me a clear picture of the class imbalance I need to deal with during modelling.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
counts = df['Churn'].value_counts()
ax.bar(['No Churn (0)', 'Churn (1)'], counts.values, color=['steelblue', 'tomato'])
ax.set_title('Churn Distribution', fontsize=14, fontweight='bold')
ax.set_ylabel('Number of Customers')
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, f'{v} ({v/len(df)*100:.1f}%)', ha='center', fontsize=11)
plt.tight_layout()
plt.savefig('outputs/eda/churn_distribution.png', dpi=150)
plt.show()

**Interpretation:** Around 26.5% of customers churned. This is a meaningful imbalance — if I trained a model without addressing it, it would simply predict No Churn for everyone and still look 73.5% accurate. SMOTE oversampling will be applied during modelling to counter this.

## 5 — Correlation Heatmap

I compute Pearson correlations across all numeric features to get a quick view of which variables move together and which are most associated with Churn.

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr_matrix = numeric_df.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Correlation Heatmap — Numeric Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/eda/correlation_heatmap.png', dpi=150)
plt.show()

print('Correlation with Churn (sorted):')
print(corr_matrix['Churn'].drop('Churn').sort_values(ascending=False))

**Interpretation:** `tenure` has the strongest negative correlation with Churn (-0.35), meaning customers who have been with the company longer are much less likely to leave. `MonthlyCharges` has a positive correlation (+0.19), suggesting higher bills are linked to higher churn risk. `TotalCharges` is negatively correlated, largely because it scales with tenure.

## 6 — Tenure vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, grp in df.groupby('Churn'):
    grp['tenure'].plot(kind='hist', alpha=0.6, bins=30, ax=ax,
                       label='Churn' if label == 1 else 'No Churn')
ax.set_title('Tenure Distribution by Churn Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/eda/tenure_vs_churn.png', dpi=150)
plt.show()

**Interpretation:** Churned customers are heavily concentrated at low tenure values. This tells me that new customers are at the greatest churn risk and that early retention efforts would have the most impact.

## 7 — Monthly Charges vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, grp in df.groupby('Churn'):
    grp['MonthlyCharges'].plot(kind='hist', alpha=0.6, bins=30, ax=ax,
                               label='Churn' if label == 1 else 'No Churn')
ax.set_title('Monthly Charges Distribution by Churn Status', fontsize=13, fontweight='bold')
ax.set_xlabel('Monthly Charges ($)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('outputs/eda/monthlycharges_vs_churn.png', dpi=150)
plt.show()

**Interpretation:** Churned customers tend to pay higher monthly charges. Customers paying above approximately $65 per month show a noticeably elevated churn rate, which suggests price sensitivity is a real driver of customer loss.

## 8 — Contract Type vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
contract_churn = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)
ax.bar(contract_churn.index, contract_churn.values * 100, color='tomato')
ax.set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 100)
for i, v in enumerate(contract_churn.values):
    ax.text(i, v * 100 + 1, f'{v*100:.1f}%', ha='center')
plt.tight_layout()
plt.savefig('outputs/eda/contract_vs_churn.png', dpi=150)
plt.show()

**Interpretation:** Month-to-month customers churn at around 43% compared to just 11% for one-year and 3% for two-year contracts. Long-term contracts are clearly the single most powerful retention mechanism in this dataset.

## 9 — Tech Support vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ts_churn = df.groupby('TechSupport')['Churn'].mean().sort_values(ascending=False)
ax.bar(ts_churn.index, ts_churn.values * 100, color='steelblue')
ax.set_title('Churn Rate by Tech Support Status', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 60)
for i, v in enumerate(ts_churn.values):
    ax.text(i, v * 100 + 1, f'{v*100:.1f}%', ha='center')
plt.tight_layout()
plt.savefig('outputs/eda/techsupport_vs_churn.png', dpi=150)
plt.show()

**Interpretation:** Customers without tech support churn at roughly twice the rate of those who have it. Promoting tech support access to high-risk customers could be an effective retention strategy.

## 10 — Internet Service vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
is_churn = df.groupby('InternetService')['Churn'].mean().sort_values(ascending=False)
ax.bar(is_churn.index, is_churn.values * 100, color='mediumpurple')
ax.set_title('Churn Rate by Internet Service Type', fontsize=13, fontweight='bold')
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 60)
for i, v in enumerate(is_churn.values):
    ax.text(i, v * 100 + 1, f'{v*100:.1f}%', ha='center')
plt.tight_layout()
plt.savefig('outputs/eda/internetservice_vs_churn.png', dpi=150)
plt.show()

**Interpretation:** Fibre optic customers churn at around 42%, which is more than double the rate for DSL customers. This could reflect higher price expectations or service quality issues with the fibre tier.

---

## Conclusions

The EDA has confirmed several strong relationships with churn:

* **Tenure** is the strongest numeric predictor — low-tenure customers are at the highest risk.
* **MonthlyCharges** is positively correlated with churn — higher-paying customers are more likely to leave.
* **Contract type** is the most impactful categorical feature — month-to-month contracts drive the majority of churn.
* **TechSupport** and **InternetService** both show meaningful differences in churn rate across their categories.

These findings answer **Business Requirement 1** and will inform the feature importance analysis during modelling. All plots have been saved to `outputs/eda/`.